# Setup

In [1]:
# Setup
import json
import os
import sys
from datetime import datetime, timedelta, timezone
import time
from dotenv import load_dotenv, set_key
import random
from pathlib import Path

# hashing for signing
import hashlib
import hmac

# requests
import requests

# data munging
import pandas as pd
import numpy as np

# helper functions
from tiktok_api_helpers import *

## API Setup

In [2]:
new_access_token = False
use_refresh_token = False

In [3]:
if (new_access_token | use_refresh_token):

    # get new acces_token
    token_url = "https://auth.tiktok-shops.com/api/v2/token/get"

    if use_refresh_token:
        auth_code = os.environ.get("TIKTOK_REFRESH_TOKEN")
        
    params = {
        "app_key": app_key,
        "app_secret": app_secret,
        "auth_code": auth_code,
        "grant_type": "authorized_code",
    }
    
    response = requests.get(token_url, params=params, timeout=15)
    data = response.json()['data']
    access_token = data['access_token']
    print(data)

    # save tokens in .env file
    set_key(".env", "TIKTOK_ACCESS_TOKEN", data['access_token'])
    set_key(".env", "TIKTOK_REFRESH_TOKEN", data['refresh_token'])

else:
    access_token = os.environ.get("TIKTOK_ACCESS_TOKEN")
    refresh_token = os.environ.get("TIKTOK_REFRESH_TOKEN")

# Step 1: Load Creator Open IDs

In [4]:
# load all creators
df_creators_list = pd.read_excel("all_creators_sorted.xlsx", sheet_name="LIST_CREATOR", usecols=[1, 2, 25])
list_all_creators = df_creators_list['username'].tolist()

In [5]:
# Load or initialize the manifest (tracks which handles are already found)
manifest = pd.read_csv(MANIFEST_CSV)
df_creators = pd.read_csv(CONSOLIDATED_CSV)

# recheck manifest in case of failed runs
manifest["found"] = manifest["handle"].isin(set(df_creators['username']))
manifest.to_csv(MANIFEST_CSV, index=False)

In [6]:
manifest['found'].sum()

np.int64(5094)

# Step 2: Extract List of existing target collaborations to avoid invitation conflicts

In [7]:
# load IDs to check
creator_open_id_list = df_creators.loc[df_creators['username'].isin(set(manifest['handle'])), 'creator_open_id'].tolist()
product_id_list = ['1734810555690551128']

In [8]:
# Load known conflicts from previous runs
conflicts_manifest_df = pd.read_csv(CONFLICTS_MANIFEST_CSV, dtype=str)

known_conflict_ids = set(conflicts_manifest_df["creator_open_id"])

# Skip creators already known to conflict
creators_to_check = [c for c in creator_open_id_list if c not in known_conflict_ids]

skipped_count = len(creator_open_id_list) - len(creators_to_check)
if skipped_count:
    print(f"Skipping {skipped_count} creator(s) already known to conflict. Checking {len(creators_to_check)} potentially new creators.")

Skipping 3618 creator(s) already known to conflict. Checking 1476 potentially new creators.


In [9]:
# find all open IDs with conflicts
all_conflict_items = []

for i in range(0, len(creators_to_check), 50):
    batch = creators_to_check[i:i + 50]
    result = check_target_collaboration_conflicts(
        creator_open_id_list=batch,
        product_id_list=product_id_list,
    )

    if result.get("code") == 0:
        all_conflict_items.extend(result["data"]["conflict_items"])
    else:
        print(f"  ⚠️  Batch failed: {result}")
        print(batch)
        
df_all_collab_conflicts = pd.DataFrame(all_conflict_items)

# Merge any newly found conflicts into the manifest
if not df_all_collab_conflicts.empty:
    df_all_collab_conflicts = pd.concat([conflicts_manifest_df, df_all_collab_conflicts], ignore_index=True)
    df_all_collab_conflicts = df_all_collab_conflicts.drop_duplicates(subset=["creator_open_id"], keep="first")
    df_all_collab_conflicts.to_csv(CONFLICTS_MANIFEST_CSV, index=False)
    new_count = len(df_all_collab_conflicts) - len(conflicts_manifest_df)
    print(f"Manifest now has {len(df_all_collab_conflicts)} known conflict(s) ({new_count} new this run).")
else:
    df_all_collab_conflicts = conflicts_manifest_df

Manifest now has 3926 known conflict(s) (308 new this run).


In [10]:
df_all_collab_conflicts

,creator_open_id,existing_collaboration_id,product_id
0,sQWUmQAAAACOV4AEeERJl-yievH_HBssk7035z_rR3rfGg...,7661970332203550472,1734810555690551128
1,XoTXawAAAACOV4AEeERJl-yievH_HBssONvLAt04stNsY9...,7661970332203550472,1734810555690551128
2,fTcLggAAAACOV4AEeERJl-yievH_HBssZ2wBpc8HcDmwra...,7661970334190913301,1734810555690551128
3,sW6t0wAAAACOV4AEeERJl-yievH_HBssY7WK-VuIXdWMLi...,7661970369088603924,1734810555690551128
4,tKpu7wAAAACOV4AEeERJl-yievH_HBssgYMxHyHuweMR6j...,7661970518602401552,1734810555690551128
...,...,...,...
3921,WyCIcAAAAACOV4AEeERJl-yievH_HBssOwka6mqSuHzTy2...,7657213115356710674,1734810555690551128
3922,AJkEcwAAAACOV4AEeERJl-yievH_HBssHeC-6rD9oYRbzq...,7657213115356710674,1734810555690551128
3923,eqs1vgAAAACOV4AEeERJl-yievH_HBssjRkrf8qhTQ9haM...,7660461201453172498,1734810555690551128
3924,MwjPHAAAAACOV4AEeERJl-yievH_HBssp0zm_YKylf0ZVX...,7660461201453172498,1734810555690551128


## Review current status

In [11]:
# merge all extracted data
df_creators_list_id_conflict = df_creators_list \
    .merge(df_creators[['username', 'creator_open_id']], how='left', on="username") \
    .merge(df_all_collab_conflicts, how='left', on="creator_open_id")

In [12]:
df_creator_summary = df_creators_list_id_conflict.groupby('batch_name').agg(
    all_creators=('creator_open_id', 'size'),
    creators_with_id=('creator_open_id', 'count'),
    invited=('existing_collaboration_id', 'count'),
)

df_creator_summary['to_invite'] = df_creator_summary['creators_with_id'] - df_creator_summary['invited']

In [13]:
df_creator_summary

,all_creators,creators_with_id,invited,to_invite
batch_name,,,,
All_202606_00k-02k,1395,1337,1300,37
All_202606_02k-04k,1534,610,72,538
All_202606_04k-06k,1630,94,60,34
All_202606_06k-08k,1660,78,62,16
All_202606_08k-10k,1672,51,8,43
All_202606_10k-12k,1707,49,2,47
Health_202606_00k-02k,2000,1955,1914,41
Health_202606_02k-04k,2000,787,470,317
Health_202606_04k-06k,2000,40,11,29


## Prepare shortlisted file without conflicts for batch processing

In [14]:
# set aside all new creators with open IDs but no conflicts
df_creators_list_id_new = df_creators_list_id_conflict[df_creators_list_id_conflict['creator_open_id'].notnull() & df_creators_list_id_conflict['existing_collaboration_id'].isnull()].reset_index(drop=True).copy()

In [15]:
# check counts
num_creators_with_conflict = df_creators_list_id_conflict['existing_collaboration_id'].notnull().sum()
num_creators_with_id = df_creators_list_id_conflict['creator_open_id'].notnull().sum()

print(f"{num_creators_with_conflict} out of {num_creators_with_id} with conflicts. {df_creators_list_id_new.shape[0]} new creator_open_id remaining for new target collaborations")

3926 out of 5094 with conflicts. 1168 new creator_open_id remaining for new target collaborations


In [16]:
df_creators_list_id_new.groupby('batch_name').size()

batch_name
All_202606_00k-02k        37
All_202606_02k-04k       538
All_202606_04k-06k        34
All_202606_06k-08k        16
All_202606_08k-10k        43
All_202606_10k-12k        47
Health_202606_00k-02k     41
Health_202606_02k-04k    317
Health_202606_04k-06k     29
Health_202606_06k-08k     29
Health_202606_08k-10k     19
Health_202606_10k-12k     18
dtype: int64

# Step 3: Create new Target Collaboration in batches of 50 new creators

In [17]:
# Set Default Values
message = "Hi {{user_name}}! \n\nWe'd love to have you as a Vitami affiliate. We make PMS Relief Gummies to help women get through their period with less discomfort. You'll get 20% commission plus a free sample for 1 TikTok video. Kindly accept this invite and request your sample if you're interested so we can ship it right away. Hoping to spread the word on better period care together! \u200d\u200d"
end_time = "2101132799"
products = [{
    "id": '1734810555690551128',
    "target_commission_rate":2000, 
    "shop_ads_commission_rate": 300
}]
seller_contact_info = {
    "email": 'admin@vitamigummies.com'
}
free_sample_rule = {
    'has_free_sample': True,
    'is_sample_approval_exempt': True
}

In [18]:
# Load prior runs' manifest so chunk numbering continues from where each batch_name last left off, instead of restarting at 01 every run.
manifest_df = pd.read_csv(TARGET_COLLAB_MANIFEST_CSV)

## Batch process Target Collab

In [19]:
# create collab only if there are at N=optimize_cutoff in the group
optimize_collab_count = True
optimize_cutoff = 50

In [20]:
# Batch-create target collaborations: group by batch_name, then chunk each group's creators into groups of 50 (the API's max per invitation)
results = []
creator_rows = []  # long-format rows: one per (target_collaboration_id, creator_open_id)

for batch_name, group in df_creators_list_id_new.groupby('batch_name'):
    creator_ids = group['creator_open_id'].tolist()
    chunks = [creator_ids[i:i + 50] for i in range(0, len(creator_ids), 50)]
    # Find the highest chunk_count already used for this batch_name in the manifest (across any previous run), and continue numbering from there.
    existing_counts = manifest_df.loc[manifest_df["batch_name"] == batch_name, "chunk_count"]
    start_count = int(existing_counts.max()) + 1 if not existing_counts.empty else 1

    for offset, chunk in enumerate(chunks):
        if optimize_collab_count:
            if len(chunk) < optimize_cutoff:
                continue

        chunk_count = start_count + offset
        name = f"{batch_name}_{chunk_count:02d}"
        print(f"Creating '{name}' with {len(chunk)} creators...")

        result = create_target_collaboration(
            name=name,
            end_time=end_time,
            products=products,
            creator_user_open_ids=chunk,
            seller_contact_info=seller_contact_info,
            free_sample_rule=free_sample_rule,
            message=message,
        )
        # print(result)

        created_at = datetime.now(timezone.utc).isoformat()

        if result.get("code") == 0:
            collab_id = result["data"]["target_collaboration"]["id"]
            print(f"  ✅ Created: {collab_id}")

            # One row per creator invited to this collaboration, only recorded
            # on success (a failed collaboration never actually invited anyone).
            for creator_open_id in chunk:
                creator_rows.append({
                    "target_collaboration_id": collab_id,
                    "name": name,
                    "batch_name": batch_name,
                    "creator_open_id": creator_open_id,
                    "end_time": end_time,
                })
        else:
            collab_id = None
            print(f"  ⚠️  Failed: {result}")

        results.append({
            "name": name,
            "batch_name": batch_name,
            "chunk_count": chunk_count,
            "num_creators": len(chunk),
            "target_collaboration_id": str(collab_id),
            "code": result.get("code"),
            "message": result.get("message"),
            "created_at": created_at,
            "end_time": end_time,
        })

# Append a summary row per collaboration to the CSV manifest.
if results:
    df_results = pd.DataFrame(results)
    manifest_exists = Path(TARGET_COLLAB_MANIFEST_CSV).exists()
    df_results.to_csv(TARGET_COLLAB_MANIFEST_CSV, mode="a", header=not manifest_exists, index=False)
    print(f"\nAppended {len(df_results)} row(s) to: {TARGET_COLLAB_MANIFEST_CSV}")
else:
    print("\nNo creators to process this run.")

# Append the long-format target_collaboration_id -> creator_open_id rows.
# This file is append-only: every run just adds more rows, never rewrites
# or deduplicates existing ones (each row records a real invitation event).
if creator_rows:
    df_creator_rows = pd.DataFrame(creator_rows)
    creators_csv_exists = Path(TARGET_COLLAB_CREATORS_CSV).exists()
    df_creator_rows.to_csv(TARGET_COLLAB_CREATORS_CSV, mode="a", header=not creators_csv_exists, index=False)
    print(f"Appended {len(df_creator_rows)} row(s) to: {TARGET_COLLAB_CREATORS_CSV}")

Creating 'All_202606_02k-04k_03' with 50 creators...
  ✅ Created: 7662936172335138580
Creating 'All_202606_02k-04k_04' with 50 creators...
  ✅ Created: 7662936290603878164
Creating 'All_202606_02k-04k_05' with 50 creators...
  ✅ Created: 7662936307242649365
Creating 'All_202606_02k-04k_06' with 50 creators...
  ✅ Created: 7662936033822934804
Creating 'All_202606_02k-04k_07' with 50 creators...
  ✅ Created: 7662936069485233940
Creating 'All_202606_02k-04k_08' with 50 creators...
  ✅ Created: 7662936290858256149
Creating 'All_202606_02k-04k_09' with 50 creators...
  ✅ Created: 7662936155536918290
Creating 'All_202606_02k-04k_10' with 50 creators...
  ✅ Created: 7662935734937405191
Creating 'All_202606_02k-04k_11' with 50 creators...
  ✅ Created: 7662936042992748295
Creating 'All_202606_02k-04k_12' with 50 creators...
  ✅ Created: 7662936235115955988
Creating 'Health_202606_02k-04k_11' with 50 creators...
  ✅ Created: 7662936248592844564
Creating 'Health_202606_02k-04k_12' with 50 creator

In [21]:
manifest_df = pd.read_csv(TARGET_COLLAB_MANIFEST_CSV)

In [22]:
manifest_df['num_creators'].sum()

np.int64(4400)

In [23]:
manifest_df

,name,batch_name,chunk_count,num_creators,target_collaboration_id,code,message,created_at,end_time
0,All_202606_02k-04k_01,All_202606_02k-04k,1,21,7661970332203550472,0,Success,NaN,2101132799
1,All_202606_04k-06k_01,All_202606_04k-06k,1,10,7661970508745393940,0,Success,NaN,2101132799
2,All_202606_06k-08k_01,All_202606_06k-08k,1,12,7661970369088603924,0,Success,NaN,2101132799
3,All_202606_08k-10k_01,All_202606_08k-10k,1,7,7661970518602401552,0,Success,NaN,2101132799
4,All_202606_10k-12k_01,All_202606_10k-12k,1,2,7661970495009081108,0,Success,NaN,2101132799
...,...,...,...,...,...,...,...,...,...
91,Health_202606_02k-04k_12,Health_202606_02k-04k,12,50,7662936254091200276,0,Success,2026-07-16T01:46:48.322909+00:00,2101132799
92,Health_202606_02k-04k_13,Health_202606_02k-04k,13,50,7662936237113050901,0,Success,2026-07-16T01:46:49.318903+00:00,2101132799
93,Health_202606_02k-04k_14,Health_202606_02k-04k,14,50,7662936218541213460,0,Success,2026-07-16T01:46:50.471957+00:00,2101132799
94,Health_202606_02k-04k_15,Health_202606_02k-04k,15,50,7662936208448915217,0,Success,2026-07-16T01:46:51.432028+00:00,2101132799


# [One-time run] Extract creator-to-target_collab_id data from created target collabs

In [4]:
# # read
# manifest_df = pd.read_csv(TARGET_COLLAB_MANIFEST_CSV, encoding="utf-8-sig", dtype=str)

# # Only successful collaborations actually have a real target_collaboration_id to query.
# successful = manifest_df[manifest_df["code"] == "0"].drop_duplicates(subset="target_collaboration_id")
# print(f"{len(successful)} successful collaboration(s) to backfill out of {len(manifest_df)} manifest row(s).")

74 successful collaboration(s) to backfill out of 74 manifest row(s).


In [5]:
# creator_rows = []

# for i, row in enumerate(successful.itertuples(index=False), start=1):
#     collab_id = row.target_collaboration_id
#     print(f"{i}/{len(successful)}: querying {row.name} ({collab_id})...")

#     result = query_target_collaboration_detail(collab_id)

#     if result.get("code") != 0:
#         print(f"  ⚠️  Failed: {result}")
#     else:
#         detail = result["data"]["target_collaboration"]
#         for creator in detail.get("creators", []):
#             creator_rows.append({
#                 "target_collaboration_id": collab_id,
#                 "name": detail.get("name", row.name),
#                 "batch_name": row.batch_name,
#                 "creator_open_id": creator["creator_open_id"],
#                 "end_time": detail.get("end_time"),
#             })
#         print(f"  ✅ {len(detail.get('creators', []))} creator(s)")

#     time.sleep(DELAY_BETWEEN_QUERIES)

# if creator_rows:
#     df_creator_rows = pd.DataFrame(creator_rows)
#     df_creator_rows.to_csv(TARGET_COLLAB_CREATORS_CSV, index=False)
#     print(f"\nSaved {len(df_creator_rows)} row(s) to {TARGET_COLLAB_CREATORS_CSV}")
# else:
#     print("\nNo creator rows collected.")

1/74: querying All_202606_02k-04k_01 (7661970332203550472)...
  ✅ 21 creator(s)
2/74: querying All_202606_04k-06k_01 (7661970508745393940)...
  ✅ 10 creator(s)
3/74: querying All_202606_06k-08k_01 (7661970369088603924)...
  ✅ 12 creator(s)
4/74: querying All_202606_08k-10k_01 (7661970518602401552)...
  ✅ 7 creator(s)
5/74: querying All_202606_10k-12k_01 (7661970495009081108)...
  ✅ 2 creator(s)
6/74: querying Health_202606_02k-04k_01 (7661970334190913301)...
  ✅ 12 creator(s)
7/74: querying Health_202606_04k-06k_01 (7661970532174645000)...
  ✅ 10 creator(s)
8/74: querying Health_202606_06k-08k_01 (7661970561066288913)...
  ✅ 11 creator(s)
9/74: querying Health_202606_08k-10k_01 (7661970554702726929)...
  ✅ 9 creator(s)
10/74: querying Health_202606_10k-12k_01 (7661970538965059346)...
  ✅ 6 creator(s)
11/74: querying Health_202606_00k-02k_01 (7661985753756845845)...
  ✅ 50 creator(s)
12/74: querying Health_202606_00k-02k_02 (7661985586839930632)...
  ✅ 50 creator(s)
13/74: querying All_